# Inspect intermediate results

:::{autolink-concat}
:::

The {class}`.StateTransitionManager` does not go from a reaction description to a {class}`.ReactionInfo` object in one step. It first creates {class}`.ProblemSet`s and then solves these as sets of quantum numbers, only matching {class}`.Particle` definitions to those quantum numbers at the very end. Each of these intermediate results can be inspected and modified, which is what this page is about. See {doc}`/usage/reaction` for the workflow as a whole and {doc}`/usage/visualize` for the rendering functions that are used here.

:::{warning}
Currently the main user-interface is the {class}`.StateTransitionManager`. There is [work in progress](https://github.com/ComPWA/qrules/issues/305) to remove it and split its functionality into several functions/classes to separate concerns and to facilitate the modification of intermediate results like the filtering of {obj}`.QNProblemSet`s, setting allowed interaction types, etc. (see below).
:::

In [ ]:
from IPython.display import Markdown

import qrules
from qrules.conservation_rules import (
    parity_conservation,
    spin_magnitude_conservation,
    spin_validity,
)
from qrules.quantum_numbers import EdgeQuantumNumbers, NodeQuantumNumbers
from qrules.solving import (
    CSPSolver,
    dict_set_intersection,
    filter_quantum_number_problem_set,
)

(problem-sets)=

## {class}`.ProblemSet`s

As noted in {doc}`/usage/reaction`, the {class}`.StateTransitionManager` provides more control than the façade function {func}`.generate_transitions`. One advantages, is that the {class}`.StateTransitionManager` first generates a set of {class}`.ProblemSet`s with {meth}`.create_problem_sets` that you can further configure if you wish.

In [ ]:
from qrules.settings import InteractionType

stm = qrules.StateTransitionManager(
    initial_state=["J/psi(1S)"],
    final_state=["K0", "Sigma+", "p~"],
    formalism="canonical-helicity",
)
stm.set_allowed_interaction_types([InteractionType.STRONG, InteractionType.EM])
problem_sets = stm.create_problem_sets()

Note that the output of {meth}`.create_problem_sets` is a {obj}`dict` with {obj}`float` values as keys (representing the interaction strength) and {obj}`list`s of {obj}`.ProblemSet`s as values.

In [ ]:
sorted(problem_sets, reverse=True)

In [ ]:
problem_set = problem_sets[60.0][0]

A {class}`.ProblemSet` can be rendered with Mermaid as follows:

In [ ]:
source = qrules.io.asmermaid(problem_set, markdown=True, render_node=True)
Markdown(source)

## Quantum number solutions

As noted in {ref}`usage/reaction:3. Find solutions`, a {obj}`.ProblemSet` can be fed to {meth}`.StateTransitionManager.find_solutions` directly to get a {obj}`.ReactionInfo` object. {obj}`.ReactionInfo` is a final result that consists of {obj}`.Particle`s, but in the intermediate steps, QRules works with sets of quantum numbers. One can inspect these intermediate generated quantum numbers by using {meth}`.find_quantum_number_transitions` and inspecting is output. Note that the resulting object is again a {obj}`dict` with strengths as keys and a list of solution as values.

In [ ]:
qn_solutions = stm.find_quantum_number_transitions(problem_sets)
{strength: len(values) for strength, values in qn_solutions.items()}

The list of solutions consists of a {obj}`tuple` of a {obj}`.QNProblemSet` (compare {ref}`problem-sets`) and a {obj}`.QNResult`:

In [ ]:
strong_qn_solutions = qn_solutions[3600.0]
qn_problem_set, qn_result = strong_qn_solutions[0]

In [ ]:
source = qrules.io.asmermaid(qn_problem_set, markdown=True, render_node=True)
Markdown(source)

In [ ]:
source = qrules.io.asmermaid(qn_result, markdown=True, render_node=True)
Markdown(source)

### Filtering quantum number problem sets

Sometimes, only a certain subset of quantum numbers and conservation rules are relevant, or the number of solutions the {class}`.StateTransitionManager` gives by default is too large for the follow-up analysis.
The {func}`.filter_quantum_number_problem_set` function can be used to produce a {class}`.QNProblemSet` where only the desired quantum numbers and conservation rules are considered when fed back to the solver.

In [ ]:
desired_edge_properties = {
    EdgeQuantumNumbers.spin_magnitude,
    EdgeQuantumNumbers.parity,
}
filtered_qn_problem_set = filter_quantum_number_problem_set(
    qn_problem_set,
    edge_rules={spin_validity},
    node_rules={spin_magnitude_conservation, parity_conservation},
    edge_properties=desired_edge_properties,
    node_properties={
        NodeQuantumNumbers.l_magnitude,
        NodeQuantumNumbers.s_magnitude,
    },
)

In [ ]:
source = qrules.io.asmermaid(filtered_qn_problem_set, markdown=True, render_node=True)
Markdown(source)

:::{warning}
The next cell will use some (currently) internal functionality. As stated at the top, a workflow similar to this will be used in future versions of {mod}`qrules`. Manual setup of the {obj}`.CSPSolver` like in here will then also not be necessary.
:::

In [ ]:
solver = CSPSolver([
    dict_set_intersection(
        qrules.system_control.create_edge_properties(part),
        desired_edge_properties,
    )
    for part in qrules.particle.load_pdg()
])
filtered_qn_solutions = solver.find_solutions(filtered_qn_problem_set)
filtered_qn_result = filtered_qn_solutions.solutions[6]

In [ ]:
source = qrules.io.asmermaid(filtered_qn_result, markdown=True, render_node=True)
Markdown(source)

:::{seealso}
[](../visualize.ipynb)
:::